# FLUX Weight Injection — Complete Native Components

**Purpose:** Inject trained weights from phase checkpoints into Flux-Apex-V1.flx

**Target Version:** v8.1-complete

**Reference:** `output/flux_native_results/README.md`

## What This Notebook Does

1. **Download** only the main .flx model (phase checkpoints downloaded on-demand)
2. **Load** Flux-Apex-V1.flx (v8.0-autonomous)
3. **Audit** current component status (identify empty/config-only)
4. **Stream inject** — download each phase checkpoint, inject, delete to save disk
5. **Verify** injected components have proper weight counts
6. **Save** as v8.1-complete (deletes original first to ensure enough disk space)

**Disk-efficient:** Only needs ~17GB free (vs ~22GB if all downloaded at once)

## Components to Inject

| Phase | Component | Current Status | Source Checkpoint |
|-------|-----------|----------------|-------------------|
| 1.5 | Causal Wave Chaining | Not present | `phase1.5.phase.pt` |
| 3 | Gravitational Relevance | Config only | `phase3.phase.pt` |
| 4 | Thermodynamic Learning | Config only | `phase4.phase.pt` |
| 5 | CGN Causal Graph | **EMPTY (0 params!)** | `phase5.phase.pt` |
| 6 | Three-Tier Memory | Metadata only | `phase6.phase.pt` |

After injection, ALL native FLUX components will have trained weights.

In [ ]:
# Cell 1: Setup and Imports
import sys
import os
import gc
import torch
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional

# Detect environment
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
ON_COLAB = 'COLAB_GPU' in os.environ

if ON_KAGGLE:
    # Clone repo if on Kaggle
    !git clone https://github.com/Unseengap/FLUX.git /kaggle/working/flux 2>/dev/null || git -C /kaggle/working/flux pull
    os.chdir('/kaggle/working/flux')
    ROOT = Path('/kaggle/working/flux')
elif ON_COLAB:
    !git clone https://github.com/Unseengap/FLUX.git /content/flux 2>/dev/null || git -C /content/flux pull
    os.chdir('/content/flux')
    ROOT = Path('/content/flux')
else:
    # Local environment
    ROOT = Path('.').resolve()
    if not (ROOT / 'flux_model.py').exists():
        ROOT = ROOT.parent
    os.chdir(ROOT)

sys.path.insert(0, str(ROOT))

print(f"✓ Working directory: {ROOT}")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Download ONLY the main model first (phase checkpoints downloaded on-demand)
from huggingface_hub import hf_hub_download, list_repo_files

HF_REPO = "UnseenGAP/FLUX"
CHECKPOINTS_DIR = ROOT / 'checkpoints'
CHECKPOINTS_DIR.mkdir(exist_ok=True)

# Get HF token
HF_TOKEN = None
try:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    else:
        HF_TOKEN = os.environ.get('HF_TOKEN')
    if HF_TOKEN:
        print("✓ HF_TOKEN loaded")
except:
    print("⚠ No HF_TOKEN — downloads may be rate-limited")

# List available files in repo
print(f"\n📂 Listing files in {HF_REPO}...")
try:
    files = list_repo_files(HF_REPO, token=HF_TOKEN)
    checkpoint_files = [f for f in files if f.startswith('checkpoints/') and (f.endswith('.pt') or f.endswith('.flx'))]
    print(f"Available checkpoints: {len(checkpoint_files)}")
    for f in checkpoint_files:
        print(f"  {f}")
except Exception as e:
    print(f"⚠ Could not list files: {e}")

# Download ONLY the main .flx file (phase checkpoints will be downloaded on-demand to save disk)
print(f"\n📥 Downloading main model only (phase checkpoints downloaded on-demand)...")
APEX_PATH = CHECKPOINTS_DIR / 'Flux-Apex-V1.flx'

if APEX_PATH.exists():
    size_mb = APEX_PATH.stat().st_size / 1e6
    print(f"  ✓ Flux-Apex-V1.flx (cached, {size_mb:.1f} MB)")
else:
    path = hf_hub_download(
        repo_id=HF_REPO,
        filename='checkpoints/Flux-Apex-V1.flx',
        local_dir=ROOT,
        token=HF_TOKEN,
    )
    size_mb = Path(path).stat().st_size / 1e6
    print(f"  ✓ Flux-Apex-V1.flx ({size_mb:.1f} MB)")

# Check disk space
import shutil
total, used, free = shutil.disk_usage(CHECKPOINTS_DIR)
print(f"\n💾 Disk space: {free/1e9:.1f} GB free")

In [ ]:
# Cell 3: Load Flux-Apex Model and Audit Current State
from flux_model import FLUXModel

APEX_PATH = CHECKPOINTS_DIR / 'Flux-Apex-V1.flx'
print(f"📂 Loading {APEX_PATH}...")

model = FLUXModel.load(str(APEX_PATH))

print(f"\n{'='*60}")
print(f"FLUX-APEX-V1 CURRENT STATE")
print(f"{'='*60}")
print(f"Version: {model.version}")
print(f"Phase: {model.phase}")
print(f"Format: {model.state.get('format', 'FLUX')}")

# Count parameters per component
def count_params(obj, depth=0):
    """Recursively count parameters in a nested dict/tensor structure."""
    if isinstance(obj, torch.Tensor):
        return obj.numel()
    if isinstance(obj, dict):
        total = 0
        for k, v in obj.items():
            total += count_params(v, depth+1)
        return total
    if isinstance(obj, (list, tuple)):
        return sum(count_params(x, depth+1) for x in obj)
    return 0

print(f"\n📊 Component Parameter Counts:")
print(f"{'Component':<25} {'Params':>15} {'Status':<20}")
print("-" * 60)

components_status = {}
for key in sorted(model.state.keys()):
    if key in ['format', 'version', 'phase', 'timestamp', 'runtime_config', 
               'components', 'metadata', 'can_continue_learning', 'modified',
               'modified_components', 'removed_components', 'state']:
        continue
    
    params = count_params(model.state.get(key, {}))
    
    # Determine status
    if params == 0:
        status = "❌ EMPTY"
    elif params < 1000:
        status = "⚠️ Config only"
    else:
        status = "✅ Has weights"
    
    components_status[key] = {'params': params, 'status': status}
    print(f"{key:<25} {params:>15,} {status:<20}")

# Identify what needs injection
print(f"\n{'='*60}")
print(f"COMPONENTS NEEDING INJECTION")
print(f"{'='*60}")

INJECTION_TARGETS = {
    'causal_wave_chaining': 'phase1.5.phase.pt',
    'gravity': 'phase3.phase.pt',
    'thermodynamic': 'phase4.phase.pt',
    'causal': 'phase5.phase.pt',
    'memory': 'phase6.phase.pt',
}

for comp, ckpt in INJECTION_TARGETS.items():
    status = components_status.get(comp, {}).get('status', '❓ Missing')
    params = components_status.get(comp, {}).get('params', 0)
    print(f"  {comp}: {params:,} params → inject from {ckpt}")

In [ ]:
# Cell 4: Streaming Injection — Download, Inject, Delete (Disk-Efficient)
# This cell downloads each phase checkpoint, injects weights, then deletes to save disk space

def download_inject_delete(phase_name: str, hf_filename: str, inject_fn):
    """Download a checkpoint, inject weights, delete to save disk."""
    local_path = CHECKPOINTS_DIR / hf_filename.split('/')[-1]
    
    print(f"\n{'='*60}")
    print(f"PHASE {phase_name}: Download → Inject → Delete")
    print(f"{'='*60}")
    
    # Download
    print(f"📥 Downloading {hf_filename}...")
    try:
        path = hf_hub_download(
            repo_id=HF_REPO,
            filename=hf_filename,
            local_dir=ROOT,
            token=HF_TOKEN,
        )
        size_mb = Path(path).stat().st_size / 1e6
        print(f"  Downloaded: {size_mb:.1f} MB")
    except Exception as e:
        print(f"  ✗ Download failed: {e}")
        return False
    
    # Load
    print(f"📂 Loading checkpoint...")
    ckpt = torch.load(str(local_path), map_location='cpu', weights_only=False)
    print(f"  Keys: {list(ckpt.keys())}")
    
    # Inject (custom function for each phase)
    success = inject_fn(ckpt)
    
    # Delete checkpoint to free disk
    del ckpt
    gc.collect()
    if local_path.exists():
        os.remove(local_path)
        print(f"🗑️ Deleted {hf_filename.split('/')[-1]} to free disk")
    
    # Report disk space
    total, used, free = shutil.disk_usage(CHECKPOINTS_DIR)
    print(f"💾 Disk space: {free/1e9:.1f} GB free")
    
    return success

# === PHASE 1.5: Causal Wave Chaining ===
def inject_phase_1_5(ckpt):
    cwc_state = ckpt.get('state_dict')
    if cwc_state:
        cwc_params = count_params(cwc_state)
        print(f"  CWC parameters: {cwc_params:,}")
        model.add_component(
            name='causal_wave_chaining',
            state_dict=cwc_state,
            config=ckpt.get('config', {}),
        )
        print(f"  ✓ Injected causal_wave_chaining")
        return True
    else:
        model.state['causal_wave_chaining'] = {'state_dict': ckpt, 'source': 'phase1.5.phase.pt'}
        print(f"  ⚠ Stored full checkpoint")
        return True

download_inject_delete("1.5 (CWC)", "checkpoints/phase1.5.phase.pt", inject_phase_1_5)

# === PHASE 3: Gravitational Relevance ===
def inject_phase_3(ckpt):
    gr_state = {}
    for key in ['phase3_gr_state', 'phase3_decoder_state']:
        if key in ckpt:
            content = ckpt[key]
            params = count_params(content)
            print(f"  Found {key}: {params:,} params")
            gr_state[key] = content
    
    if gr_state:
        total_params = count_params(gr_state)
        print(f"  Total GR parameters: {total_params:,}")
        model.state['gravity'] = {
            'state_dict': gr_state,
            'config': ckpt.get('config', {}),
            'source': 'phase3.phase.pt',
            'injected_at': datetime.now().isoformat(),
        }
        model.components['gravity'] = True
        print(f"  ✓ Injected gravity component")
        return True
    else:
        model.state['gravity'] = {'full_checkpoint': ckpt, 'source': 'phase3.phase.pt'}
        return True

download_inject_delete("3 (GR)", "checkpoints/phase3.phase.pt", inject_phase_3)

# === PHASE 4: Thermodynamic Learning ===
def inject_phase_4(ckpt):
    tl_state = ckpt.get('tl_state')
    if tl_state:
        params = count_params(tl_state)
        print(f"  Found tl_state: {params:,} params")
        model.state['thermodynamic'] = {
            'state_dict': {'tl_state': tl_state},
            'config': ckpt.get('config', {}),
            'source': 'phase4.phase.pt',
            'injected_at': datetime.now().isoformat(),
        }
        model.components['thermodynamic'] = True
        print(f"  ✓ Injected thermodynamic component")
        return True
    else:
        model.state['thermodynamic'] = {'full_checkpoint': ckpt, 'source': 'phase4.phase.pt'}
        return True

download_inject_delete("4 (TL)", "checkpoints/phase4.phase.pt", inject_phase_4)

# === PHASE 5: CGN Causal Graph (CRITICAL) ===
def inject_phase_5(ckpt):
    cgn_state = {}
    for key in ['cgn_state', 'tl_state']:  # cgn_state has 14.7M, tl_state has 135M
        if key in ckpt:
            content = ckpt[key]
            params = count_params(content)
            if params > 0:
                print(f"  Found {key}: {params:,} params")
                cgn_state[key] = content
    
    if cgn_state:
        total_params = count_params(cgn_state)
        print(f"  Total CGN parameters: {total_params:,}")
        model.state['causal'] = {
            'state_dict': cgn_state,
            'config': ckpt.get('config', {}),
            'source': 'phase5.phase.pt',
            'injected_at': datetime.now().isoformat(),
        }
        model.components['causal'] = True
        model.components['causal_tracker'] = True
        print(f"  ✓ Injected causal (CGN) — was EMPTY, now has {total_params:,} params!")
        return True
    else:
        model.state['causal'] = {'full_checkpoint': ckpt, 'source': 'phase5.phase.pt'}
        return True

download_inject_delete("5 (CGN)", "checkpoints/phase5.phase.pt", inject_phase_5)

# === PHASE 6: Three-Tier Memory (CRITICAL) ===
def inject_phase_6(ckpt):
    memory_state = {}
    for key in ['working_memory_state', 'episodic_memory_state', 'semantic_memory_state', 'router_state']:
        if key in ckpt:
            content = ckpt[key]
            params = count_params(content)
            print(f"  Found {key}: {params:,} params")
            memory_state[key] = content
    
    if memory_state:
        total_params = count_params(memory_state)
        print(f"  Total Memory parameters: {total_params:,}")
        model.state['memory'] = {
            'state_dict': memory_state,
            'config': ckpt.get('config', {}),
            'working': memory_state.get('working_memory_state', {}),
            'episodic': memory_state.get('episodic_memory_state', {}),
            'semantic': memory_state.get('semantic_memory_state', {}),
            'router': memory_state.get('router_state', {}),
            'source': 'phase6.phase.pt',
            'injected_at': datetime.now().isoformat(),
        }
        model.components['memory'] = True
        print(f"  ✓ Injected memory component")
        return True
    else:
        model.state['memory'] = {'full_checkpoint': ckpt, 'source': 'phase6.phase.pt'}
        return True

download_inject_delete("6 (Memory)", "checkpoints/phase6.phase.pt", inject_phase_6)

print(f"\n{'='*60}")
print(f"✅ ALL INJECTIONS COMPLETE")
print(f"{'='*60}")

In [ ]:
# Cell 5: Update Version and Metadata
print(f"\n{'='*60}")
print(f"UPDATING VERSION AND METADATA")
print(f"{'='*60}")

# Update version
old_version = model.version
model.version = '8.1-complete'
model.phase = 'complete'

# Update metadata
model.metadata['last_modified'] = datetime.now().isoformat()
model.metadata['modified_components'] = [
    'causal_wave_chaining',
    'gravity',
    'thermodynamic',
    'causal',
    'memory',
]
model.metadata['injection_date'] = datetime.now().isoformat()
model.metadata['injection_source'] = 'flux_weight_injection.ipynb'
model.metadata['previous_version'] = old_version

# Track all injections
model.metadata['injections'] = {
    'causal_wave_chaining': 'phase1.5.phase.pt',
    'gravity': 'phase3.phase.pt',
    'thermodynamic': 'phase4.phase.pt',
    'causal': 'phase5.phase.pt',
    'memory': 'phase6.phase.pt',
}

print(f"Version: {old_version} → {model.version}")
print(f"Phase: {model.phase}")
print(f"Modified components: {model.metadata['modified_components']}")

In [ ]:
# Cell 6: Verify Injection — Parameter Count Comparison
print(f"\n{'='*60}")
print(f"VERIFICATION: Parameter Count Comparison")
print(f"{'='*60}")

print(f"\n{'Component':<25} {'Before':>15} {'After':>15} {'Change':<15}")
print("-" * 70)

after_status = {}
for key in sorted(model.state.keys()):
    if key in ['format', 'version', 'phase', 'timestamp', 'runtime_config', 
               'components', 'metadata', 'can_continue_learning', 'modified',
               'modified_components', 'removed_components', 'state']:
        continue
    
    params = count_params(model.state.get(key, {}))
    after_status[key] = params
    
    before = components_status.get(key, {}).get('params', 0)
    
    if params > before:
        change = f"✅ +{params - before:,}"
    elif params == before:
        change = "—"
    else:
        change = f"⚠️ {params - before:,}"
    
    print(f"{key:<25} {before:>15,} {params:>15,} {change:<15}")

# Summary
print(f"\n{'='*60}")
print(f"INJECTION SUMMARY")
print(f"{'='*60}")

total_before = sum(components_status.get(k, {}).get('params', 0) for k in after_status.keys())
total_after = sum(after_status.values())

print(f"Total parameters before: {total_before:,}")
print(f"Total parameters after:  {total_after:,}")
print(f"Parameters added:        {total_after - total_before:,}")

In [ ]:
# Cell 7: Save Injected Model (Disk-Safe)

# Delete original .flx BEFORE save to ensure enough disk space
SAVE_PATH = CHECKPOINTS_DIR / 'Flux-Apex-V1.flx'

print(f"\n{'='*60}")
print(f"SAVING INJECTED MODEL")
print(f"{'='*60}")

# Check initial disk space
import shutil
total, used, free = shutil.disk_usage(CHECKPOINTS_DIR)
free_gb = free / 1e9
print(f"Free disk space: {free_gb:.1f} GB")

# Delete original to free up ~15GB before saving
if SAVE_PATH.exists():
    original_size = SAVE_PATH.stat().st_size / 1e9
    print(f"Deleting original .flx ({original_size:.1f} GB) to make room...")
    os.remove(SAVE_PATH)
    gc.collect()
    total, used, free = shutil.disk_usage(CHECKPOINTS_DIR)
    free_gb = free / 1e9
    print(f"After deletion: {free_gb:.1f} GB free")

# Save
print(f"Saving to: {SAVE_PATH}")
model.save(str(SAVE_PATH), overwrite=True)

# Verify
if SAVE_PATH.exists():
    size_gb = SAVE_PATH.stat().st_size / 1e9
    print(f"✓ Saved: {SAVE_PATH.name} ({size_gb:.2f} GB)")
else:
    print(f"✗ Save failed!")

In [ ]:
# Cell 8: Final Verification — Reload and Check
print(f"\n{'='*60}")
print(f"FINAL VERIFICATION")
print(f"{'='*60}")

# Reload to verify
del model
gc.collect()

verify_model = FLUXModel.load(str(SAVE_PATH))

print(f"Version: {verify_model.version}")
print(f"Phase: {verify_model.phase}")

print(f"\n📊 Verified Component Parameters:")
print(f"{'Component':<25} {'Params':>15} {'Status':<20}")
print("-" * 60)

critical_ok = True
for key in ['cse', 'field', 'causal', 'memory', 'gravity', 'thermodynamic']:
    if key in verify_model.state:
        params = count_params(verify_model.state[key])
        status = "✅ Has weights" if params > 0 else "❌ EMPTY"
        if key in ['causal', 'memory'] and params == 0:
            critical_ok = False
        print(f"{key:<25} {params:>15,} {status:<20}")

print(f"\n{'='*60}")
if critical_ok:
    print("✅ ALL CRITICAL COMPONENTS HAVE WEIGHTS")
else:
    print("⚠️ SOME CRITICAL COMPONENTS STILL EMPTY")
print(f"{'='*60}")

In [ ]:
# Cell 9: Upload to HuggingFace (Optional)
UPLOAD_TO_HF = False  # Set to True to upload

if UPLOAD_TO_HF and HF_TOKEN:
    print(f"\n{'='*60}")
    print(f"UPLOADING TO HUGGINGFACE")
    print(f"{'='*60}")
    
    from huggingface_hub import HfApi
    
    api = HfApi(token=HF_TOKEN)
    
    # Use model.version (from Cell 10) if verify_model doesn't exist
    upload_version = verify_model.version if 'verify_model' in dir() else model.version
    
    api.upload_file(
        path_or_fileobj=str(SAVE_PATH),
        path_in_repo="checkpoints/Flux-Apex-V1.flx",
        repo_id=HF_REPO,
        commit_message=f"Inject trained weights - v{upload_version}",
    )
    
    print(f"✓ Uploaded to {HF_REPO}")
    print(f"  https://huggingface.co/{HF_REPO}")
else:
    print(f"\n⚠ Upload skipped (UPLOAD_TO_HF={UPLOAD_TO_HF})")
    print(f"  Set UPLOAD_TO_HF = True in Cell 9 to upload")

## Summary

This notebook uses **streaming injection** to minimize disk usage:
- Downloads each phase checkpoint on-demand
- Injects weights into the model
- Immediately deletes the checkpoint to free disk space
- Only ~17GB free needed (vs ~22GB if all downloaded at once)

After running:

1. **Phase 1.5 (CWC)** — Causal Wave Chaining weights injected
2. **Phase 3 (GR)** — Gravitational Relevance weights injected  
3. **Phase 4 (TL)** — Thermodynamic Learning weights injected
4. **Phase 5 (CGN)** — Causal Geometry Nodes weights injected (was EMPTY!)
5. **Phase 6 (Memory)** — Three-Tier Memory weights injected

The model is now **v8.1-complete** with ALL native FLUX components having trained weights.

3. Upload to HuggingFace when ready

### Next Steps2. Verify all tests pass with injected components

1. Run on Kaggle to perform the injection with GPU